# B3.4 · Eğitilmiş modeli Ollama'ya hazırla

**Bu defter GPU İSTEMEZ.** Yukarıdan `Çalıştırma zamanı → Tümünü çalıştır`
de ve bırak. Hiçbir hücreyi atlaman gerekmiyor.

GPU kotası doluysa bağlanırken **"GPU olmadan bağlan"** de, sorun olmaz.

## Ne yapıyor

```
1. Drive'i baglar
2. Gerekli paketleri kurar          ~3 dk
3. Egitilmis LoRA'yi taban modelle birlestirir   ~5 dk
4. GGUF'a cevirir (q8_0, ~1,6 GB)   ~5 dk
5. Drive'a kopyalar                 ~2 dk
```

Toplam ~15 dakika, çoğu indirme.

## Bitince

Drive'daki `codifya/cikti/` klasöründen `.gguf` dosyasını indir, projedeki
`training/` klasörüne koy ve yerelde şunu çalıştır:

```bash
ollama create codifya-router:tur3 -f training/Modelfile
```

## 1. Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Kurulum (~3 dk)

In [ ]:
!pip install -q peft transformers accelerate sentencepiece protobuf
!git clone --depth 1 -q https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
print('kurulum tamam')

## 3. Adaptörü bul

⚠️ Klasör adı `b33_tur1_lora` ama **içindeki 2. turun adaptörü** — 2. tur
kaydedilirken değişken adı güncellenmemiş, dosya üzerine yazılmış. Kişi A
ile teyit edildi.

Aşağıdaki liste tarih ve boyutu gösteriyor; en yeni olan doğru olandır.

In [ ]:
from datetime import datetime
from pathlib import Path

CIKTI = Path('/content/drive/MyDrive/codifya/cikti')
TUR_ETIKETI = 'tur3'   # <-- hangi turu GGUF'a cevirecegiz
LORA_YOLU = CIKTI / f'b33_{TUR_ETIKETI}_lora'
ETIKET = TUR_ETIKETI

for d in sorted(CIKTI.iterdir()):
    a = d / 'adapter_model.safetensors' if d.is_dir() else None
    if a is None or not a.exists():
        continue
    t = datetime.fromtimestamp(a.stat().st_mtime)
    print(f'  {d.name:24s} {a.stat().st_size / 1024**2:7.1f} MB   {t:%Y-%m-%d %H:%M}')

print()
assert (LORA_YOLU / 'adapter_model.safetensors').exists(), f'adapter yok: {LORA_YOLU}'
print('kullanilacak:', LORA_YOLU)

## 4. Birleştir (~5 dk)

LoRA küçük ek matrisler tutuyor; Ollama tam model istiyor. Birleştirme bu
matrisleri taban ağırlıklara işliyor.

⚠️ 4-bit **kullanılmıyor** — birleştirme tam duyarlıkla yapılmalı, yoksa
nicemleme hatası kalıcı olarak ağırlıklara girer.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

TABAN = 'Qwen/Qwen2.5-1.5B-Instruct'
BIRLESIK = Path('/content/birlesik')

print('taban model indiriliyor...')
taban = AutoModelForCausalLM.from_pretrained(TABAN, torch_dtype=torch.float16, device_map='cpu')
tokenizer = AutoTokenizer.from_pretrained(TABAN)

print('LoRA birlestiriliyor...')
model = PeftModel.from_pretrained(taban, str(LORA_YOLU))
model = model.merge_and_unload()

model.save_pretrained(str(BIRLESIK), safe_serialization=True)
tokenizer.save_pretrained(str(BIRLESIK))

gb = sum(f.stat().st_size for f in BIRLESIK.rglob('*') if f.is_file()) / 1024**3
print(f'birlestirildi: {BIRLESIK}  ({gb:.2f} GB)')

## 5. GGUF'a çevir (~5 dk)

`q8_0` seçildi: derleme gerektirmiyor, 1,5B modelde kalite kaybı
ölçülemeyecek kadar küçük, dosya ~1,6 GB.

In [ ]:
import os

GGUF = f'/content/codifya-{ETIKET}-q8_0.gguf'

!python /content/llama.cpp/convert_hf_to_gguf.py /content/birlesik --outfile {GGUF} --outtype q8_0

print()
print('GGUF :', GGUF)
print('boyut:', round(os.path.getsize(GGUF) / 1024**3, 2), 'GB')

## 6. Drive'a kopyala (~2 dk)

In [ ]:
import shutil

hedef = CIKTI / os.path.basename(GGUF)
shutil.copy(GGUF, hedef)

print('=' * 62)
print('BITTI')
print('=' * 62)
print('Dosya:', hedef)
print('Boyut:', round(hedef.stat().st_size / 1024**3, 2), 'GB')
print()
print('SIRADA:')
print('  1. Drive > codifya > cikti klasorunden bu dosyayi indir')
print('  2. D:/Codifiya-ERP/training/ klasorune koy')
print('  3. Yerelde:  ollama create codifya-router:tur3 -f training/Modelfile')